In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import scipy.sparse as sp
import anndata as ad
import gseapy as gp
from copy import deepcopy
import scanpy.external as sce
import pickle

# sc.logging.print_version()
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5,5))

import sys
from pathlib import Path

_start = Path.cwd().resolve()
_repo_root = next((p for p in [_start, *_start.parents] if (p / "chronopack").exists()), _start)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from chronopack.tradeseq import (
    ComputeSmoothCorrelations,
    TemporalPseudobulkFromAnnData,
    WriteTradeSeqInputsFromAnnData,
    load_negative_control_logcpm,
    smooth_series,
)
from chronopack.preprocessing import filter_technical_genes


/new-stg/home/alex/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /new-stg/home/alex/.tmp/matplotlib-4f8huwmy because there was an issue with the default path (/new-stg/home/alex/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
adata_hvg = sc.read_h5ad('../output/adata_hvg.chk1.h5ad')
adata = sc.read_h5ad('../output/adata_full.chk1.h5ad')

# Compute CPM & logCPM matrices for negative bulk control

In [3]:
mtx_agg, cpm_ref, logcpm_ref = load_negative_control_logcpm('../intmd/k562_neg-ctrl_mtx-agg.pkl')


# Compute pseudobulk CPM & logCPM matrices for LPS+THP1 experiment

In [4]:
cpm, logcpm = TemporalPseudobulkFromAnnData(adata)


# Prioritize genes by variation over time

In [5]:
def RenameClusters(x):
    if (x == '0'): return 'K562'
    if (x == '1'): return 'K562'
    if (x == '2'): return 'THP1'

adata.obs['cell_type'] = adata.obs['traj_clusters'].apply(RenameClusters)

# New approach

In [6]:
adata_thp1_hvg = adata[(adata.obs['cell_type'] == 'THP1'), :].copy()
adata_k562_hvg = adata[(adata.obs['cell_type'] == 'K562'), :].copy()

In [7]:
cpm_thp1, logcpm_thp1 = TemporalPseudobulkFromAnnData(adata_thp1_hvg)
cpm_k562, logcpm_k562 = TemporalPseudobulkFromAnnData(adata_k562_hvg)

In [8]:
adata_k562_hvg.var["total_counts"] = np.asarray(
    adata_k562_hvg.layers["counts"].sum(axis=0)
).ravel()

In [9]:
adata_thp1_hvg.var["total_counts"] = np.asarray(
    adata_thp1_hvg.layers["counts"].sum(axis=0)
).ravel()

In [10]:
pd.Series(adata_k562_hvg.var.index[adata_k562_hvg.var['total_counts'] < 10]).to_csv('exclude_low-count-genes.k562.csv', index=False)
pd.Series(adata_thp1_hvg.var.index[adata_thp1_hvg.var['total_counts'] < 10]).to_csv('exclude_low-count-genes.thp1.csv', index=False)

In [11]:
adata_hvg = adata.copy()
shared_genes = list(set(adata_hvg.var.index) & set(logcpm_ref.columns) & set(logcpm.columns))

cpm_ref = cpm_ref.loc[:, shared_genes]
logcpm_ref = logcpm_ref.loc[:, shared_genes]

cpm = cpm.loc[:, shared_genes]
logcpm = logcpm.loc[:, shared_genes]
logcpm.shape

(12, 19433)

# Compute correlations

In [12]:
corr_k562 = ComputeSmoothCorrelations(
    logcpm_k562,
    logcpm_ref,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.8,
    local_fraction_threshold=0.5,
)

corr_thp1 = ComputeSmoothCorrelations(
    logcpm_thp1,
    logcpm_ref,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.7,
    local_fraction_threshold=0.3,
)


In [13]:
# Post hoc re-filter corr without recomputing trajectories

local_r_threshold = 0.7
local_fraction_threshold = 0.25

corr_k562["tag_bias_candidate_residual"] = ((corr_k562["pearson_r_residual"] >= local_r_threshold) & (corr_k562["residual_explained_total_fraction"] >= local_fraction_threshold))
corr_k562["tag_bias_candidate_diff"] = ((corr_k562["pearson_r_diff"] >= local_r_threshold) & (corr_k562["diff_residual_explained_fraction"] >= local_fraction_threshold))
# corr_k562["tag_bias_candidate"] = (corr_k562["tag_bias_candidate_residual"] | corr_k562["tag_bias_candidate_diff"])
corr_k562["tag_bias_candidate"] = corr_k562["tag_bias_candidate_residual"]

print("Tag-bias candidates:", int(corr_k562["tag_bias_candidate"].sum()))

# Optional: continuous score + re-sort
corr_k562["tag_bias_score"] = corr_k562[["residual_explained_total_fraction", "diff_residual_explained_fraction",]].max(axis=1, skipna=True)
corr_k562 = corr_k562.sort_values("max_r_local", ascending=False,)


corr_k562[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)

Tag-bias candidates: 857


,pearson_r_residual,residual_explained_total_fraction,pearson_r_diff,diff_residual_explained_fraction,tag_bias_candidate_residual,tag_bias_candidate_diff,tag_bias_candidate
gene,,,,,,,
DDX25,1.0,0.500095,1.0,0.759077,True,True,True
ENSG00000279118,1.0,0.499640,1.0,0.751897,True,True,True
KCNG1,1.0,0.263736,1.0,0.521177,True,True,True
LINC02029,1.0,0.499994,1.0,0.674177,True,True,True
TIPINP2,1.0,0.263736,1.0,0.521177,True,True,True
ENSG00000287159,1.0,0.499547,1.0,0.751198,True,True,True
ENSG00000309556,1.0,0.500012,1.0,0.767349,True,True,True
LRRC34,1.0,0.499547,1.0,0.751198,True,True,True
GRAMD1B,1.0,0.500095,1.0,0.759077,True,True,True


In [14]:
# Post hoc re-filter corr without recomputing trajectories

local_r_threshold = 0.7
local_fraction_threshold = 0.25

corr_thp1["tag_bias_candidate_residual"] = ((corr_thp1["pearson_r_residual"] >= local_r_threshold) & (corr_thp1["residual_explained_total_fraction"] >= local_fraction_threshold))
corr_thp1["tag_bias_candidate_diff"] = ((corr_thp1["pearson_r_diff"] >= local_r_threshold) & (corr_thp1["diff_residual_explained_fraction"] >= local_fraction_threshold))
# corr_thp1["tag_bias_candidate"] = (corr_thp1["tag_bias_candidate_residual"] | corr_thp1["tag_bias_candidate_diff"])
corr_thp1["tag_bias_candidate"] = corr_thp1["tag_bias_candidate_residual"]


# Optional: continuous score + re-sort
corr_thp1["tag_bias_score"] = corr_thp1[["residual_explained_total_fraction", "diff_residual_explained_fraction",]].max(axis=1, skipna=True)
corr_thp1 = corr_thp1.sort_values("max_r_local", ascending=False,)

print("Tag-bias candidates:", int(corr_thp1["tag_bias_candidate"].sum()))

corr_thp1[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)

Tag-bias candidates: 507


,pearson_r_residual,residual_explained_total_fraction,pearson_r_diff,diff_residual_explained_fraction,tag_bias_candidate_residual,tag_bias_candidate_diff,tag_bias_candidate
gene,,,,,,,
EML4-AS1,1.0,0.263736,1.0,0.521177,True,True,True
ENSG00000297147,1.0,0.499928,1.0,0.741651,True,True,True
P4HA3,1.0,0.500012,1.0,0.767349,True,True,True
ENSG00000304531,1.0,0.499994,1.0,0.674177,True,True,True
DDX25,1.0,0.500095,1.0,0.759077,True,True,True
ENSG00000287159,1.0,0.499547,1.0,0.751198,True,True,True
AP4B1-AS1,1.0,0.263736,1.0,0.521177,True,True,True
LINC01206,1.0,0.500097,1.0,0.755222,True,True,True
SMARCE1P3,1.0,0.500095,1.0,0.759077,True,True,True


In [15]:
excl_genes_k562 = set(corr_k562[corr_k562['tag_bias_candidate'] == True].index.tolist())
print(len(excl_genes_k562))
print('HNRNPA1P27' in excl_genes_k562)

857
True


In [16]:
excl_genes_thp1 = set(corr_thp1[corr_thp1['tag_bias_candidate'] == True].index.tolist())
print(len(excl_genes_thp1))
print('HNRNPA1P27' in excl_genes_thp1)

507
True


# Pre-filter @K562

In [17]:
adata_k562_hvg = filter_technical_genes(adata_k562_hvg, exclude_genes=excl_genes_k562)
adata_k562_hvg


AnnData object with n_obs × n_vars = 3128 × 15847
    obs: 'COUNTS', 'sample', 'time', 'time_group', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'n_genes', 'doublet_score', 'predicted_doublet', 'S_score', 'G2M_score', 'phase', 'leiden_unintegrated', 'THP1_score', 'K562_score', 'THP1_minus_K562', 'cell_type', 'traj_leiden_sr', 'traj_clusters', 'terminal_fate_probs_traj_0', 'terminal_fate_probs_traj_1', 'terminal_fate_probs_traj_2', 'fate_0', 'fate_1', 'fate_2', 'fate_entropy', 'fate_maxprob', 'traj_clusters_mass_ret_cell'
    var: 'mt', 'rb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'hvg', 'leiden_unintegrated', 'leiden_unintegrated_colors', 'log1p', 'neighbors', 'ot_graph', 'pca', 'phase_colors', 'sample_colors', 'scrublet', 'terminal_cell_dist_idx_last', 'terminal_fate_probs_traj_fates

In [18]:
print('PLAU' in set(adata_k562_hvg.var.index)) # similar shape
print('CD83' in set(adata_k562_hvg.var.index)) # similar shape
print('EGR1' in set(adata_k562_hvg.var.index)) # similar shape
print('IER3' in set(adata_k562_hvg.var.index)) # similar shape

True
True
True
True


In [19]:
# TPs
print('REL' in set(adata_k562_hvg.var.index)) # real, should be included
print('ALAS1' in set(adata_k562_hvg.var.index)) # also real, should be included
print('SLC40A1' in set(adata_k562_hvg.var.index)) # similar shape but a lot noisier
print('TRAF3' in set(adata_k562_hvg.var.index)) # similar shape

# TNs
print('FTH1' in set(adata_k562_hvg.var.index)) # should have high correlation
print('RELA' in set(adata_k562_hvg.var.index)) # similar shape

True
True
True
True
True
True


# Pre-filter @THP1

In [20]:
adata_thp1_hvg = filter_technical_genes(adata_thp1_hvg, exclude_genes=excl_genes_thp1)
adata_thp1_hvg


AnnData object with n_obs × n_vars = 787 × 16126
    obs: 'COUNTS', 'sample', 'time', 'time_group', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'n_genes', 'doublet_score', 'predicted_doublet', 'S_score', 'G2M_score', 'phase', 'leiden_unintegrated', 'THP1_score', 'K562_score', 'THP1_minus_K562', 'cell_type', 'traj_leiden_sr', 'traj_clusters', 'terminal_fate_probs_traj_0', 'terminal_fate_probs_traj_1', 'terminal_fate_probs_traj_2', 'fate_0', 'fate_1', 'fate_2', 'fate_entropy', 'fate_maxprob', 'traj_clusters_mass_ret_cell'
    var: 'mt', 'rb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'hvg', 'leiden_unintegrated', 'leiden_unintegrated_colors', 'log1p', 'neighbors', 'ot_graph', 'pca', 'phase_colors', 'sample_colors', 'scrublet', 'terminal_cell_dist_idx_last', 'terminal_fate_probs_traj_fates'

In [21]:
print('PLAU' in set(adata_thp1_hvg.var.index)) # similar shape
print('CD83' in set(adata_thp1_hvg.var.index)) # similar shape
print('EGR1' in set(adata_thp1_hvg.var.index)) # similar shape
print('IER3' in set(adata_thp1_hvg.var.index)) # similar shape

True
True
True
True


In [22]:
# TPs
print('REL' in set(adata_thp1_hvg.var.index)) # real, should be included
print('ALAS1' in set(adata_thp1_hvg.var.index)) # also real, should be included
print('SLC40A1' in set(adata_thp1_hvg.var.index)) # similar shape but a lot noisier
print('TRAF3' in set(adata_thp1_hvg.var.index)) # similar shape

# TNs
print('FTH1' in set(adata_thp1_hvg.var.index)) # should have high correlation
print('RELA' in set(adata_thp1_hvg.var.index)) # similar shape

True
True
True
True
True
True


In [23]:
print('HNRNPA1P27' in adata_k562_hvg.var.index)
print('HNRNPA1P27' in adata_thp1_hvg.var.index)

False
False


# Prepare input files for TradeSeq (per cell type)

In [24]:
adata_full = sc.read_h5ad('../output/adata_full.chk1.h5ad')
adata_k562_full = adata_full[adata_k562_hvg.obs.index, :]
print(adata_k562_full.shape)

adata_thp1_full = adata_full[adata_thp1_hvg.obs.index, :]
print(adata_thp1_full.shape)

(3128, 21693)
(787, 21693)


# Prepare input files for TradeSeq (across cell types)

In [25]:
adata_k562_hvg.write_h5ad('../output/adata_k562_prefiltered.fixed.h5ad')
adata_thp1_hvg.write_h5ad('../output/adata_thp1_prefiltered.fixed.h5ad')


In [26]:
WriteTradeSeqInputsFromAnnData(adata_k562_hvg, 'sc-K562_to_tradeSeq.fixed')
WriteTradeSeqInputsFromAnnData(adata_thp1_hvg, 'sc-THP1_to_tradeSeq.fixed')


(15847, 3128) (3128, 2)
Wrote subset_genes.csv and cell_libsize_full.csv


(16126, 787) (787, 2)
Wrote subset_genes.csv and cell_libsize_full.csv


In [27]:
WriteTradeSeqInputsFromAnnData(adata_k562_full, 'sc-K562_to_tradeSeq.fixed', tag='full')
WriteTradeSeqInputsFromAnnData(adata_thp1_full, 'sc-THP1_to_tradeSeq.fixed', tag='full')


(21693, 3128) (3128, 2)
Wrote subset_genes.csv and cell_libsize_full.csv


(21693, 787) (787, 2)
Wrote subset_genes.csv and cell_libsize_full.csv
